In [1]:
import numpy as np
import matplotlib.pyplot as plt
from regions import Regions
import regions
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.modeling import models, fitting

from dust_extinction.averages import CT06_MWGC 

from smart_plotters.jwst_plots import JWSTCatalog, make_cat_use, make_brick_cat
from smart_plotters.cutout_plot import get_cutout_405, get_cutout_jwst_ice
import smart_plotters.co_ice as co_map
from icemodels.colorcolordiagrams import plot_ccd_icemodels
from smart_plotters import cmd_plot 
from sklearn.mixture import GaussianMixture
from mpl_plot_templates import adaptive_param_plot
from scipy.optimize import curve_fit 


        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]
/blue/adamginsburg/adamginsburg/jwst/brick-jwst-2221/brick2221/analysis/plot_tools.py:804: SyntaxWarning: invalid escape sequence '\m'
  ax.set_title(f'{filtername} $\mu$={np.median(sep).to(u.marcsec).value:0.1f} $\sigma$={np.std(sep).to(u.marcsec).value:0.1f}', fontsize=10)
/blue/adamginsburg/adamginsburg/jwst/brick-jwst-2221/brick2221/analysis/plot_tools.py:804: SyntaxWarning: invalid escape sequence '\s'
  ax.set_title(f'{filtername} $\mu$={np.median(sep).to(u.marcsec).value:0.1f} $\sigma$={np.std(sep).to(u.marcsec).value:0.1f}', fontsize=10)
/blue/adamginsburg/adamginsburg/jwst/brick-jwst-2221/brick2221/analysis/plot_tools.py:810: SyntaxWarning: invalid escape sequence '\m'
  ax2.set_title(f'{filtername} $\mu$={np.median(sep).to(u.marcsec).value:0.1f} $\sigma$={np.std(sep).to(u.marcsec).value:0.1f}', fontsize=10)
/blue/adamginsburg/adamginsburg/jwst/brick-jwst-2221/brick2221/an

In [ ]:
catalog_list = [cat_f_rc, cat_c1_rc, cat_c2_rc, cat_d_rc, cat_b_rc, cat_fil_r_rc, cat_left_cd_rc, cat_c_n_rc, cat_c_s_rc,
               cat_brick_north_rc, cat_brick_south_rc, cat_brick_head_rc, cat_brick_sfluff_rc, cat_brick_nfluff_rc, 
               cat_brick_punch_rc, cat_brick_sfore_rc, cat_brick_nclear_rc, cat_brick_sfr_rc, cat_brick_sfrwide_rc, cat_brick_topc_rc]
name_list = ['Filament', 'Cloud C1', 'Cloud C2', 'Cloud D', 'Cloud B', 'Filament Right', 'Left Cloud D', 'Cloud C North', 'Cloud C South',
             'Brick North', 'Brick South', 'Brick Head', 'Brick South Fluff', 'Brick North Fluff', 'Brick Punch', 'Brick South Foreground', 
             'Brick North Clear', 'Brick Star Forming Region', 'Brick Star Forming Region Wide', 'Brick Top Center']

mean_NCO = []
std_NCO = []
unc_NCO = []
mean_CO_abund = []
std_CO_abund = []
unc_CO_abund = []


for i in range(20):
    #ax = axes.flatten()[i]
    cat = catalog_list[i]
    nCO = cat.catalog['N(CO)'].copy()
    av = cat.catalog['Av']
    nCO[nCO <= 0] = np.nan
    nCO[cat.color('f405n', 'f466n') > 0] = np.nan
    nCO[av < 25] = np.nan
    mean_NCO.append(np.nanmean(nCO))
    std_NCO.append(np.nanstd(nCO))
    unc_NCO.append(std_NCO[-1] / np.sqrt(len(np.array(nCO))))
    mean_CO_abund.append(np.nanmean(nCO/(cat.catalog['Av']*1.1e21)))
    std_CO_abund.append(np.nanstd(nCO/(cat.catalog['Av']*1.1e21)))
    unc_CO_abund.append(std_CO_abund[-1] / np.sqrt(len(np.array(nCO))))

from astropy.table import Table
tbl = Table()
tbl['Region'] = name_list
#tbl['Mean N(CO Ice) [cm^-2] x 1e18'] = np.round(np.array(mean_NCO) / 1e18, 2)
#tbl['Std N(CO Ice) [cm^-2] x 1e18'] = np.round(np.array(std_NCO) / 1e18, 2)
#tbl['Uncertainty N(CO Ice) [cm^-2] x 1e18'] = np.round(np.array(unc_NCO) / 1e18, 2)
tbl['Mean CO Ice / H2 Abundance x 1e-4'] = np.round(np.array(mean_CO_abund) / 1e-4, 2)
tbl['Std CO Ice / H2 Abundance x 1e-4'] = np.round(np.array(std_CO_abund) / 1e-4, 2)
tbl['Uncertainty CO Ice / H2 Abundance x 1e-4'] = np.round(np.array(unc_CO_abund) / 1e-4, 2)

tbl.write('/orange/adamginsburg/jwst/cloudc/dustridge-cd/data/CO_ice_summary.tbl', format='latex', overwrite=True)